In [57]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.datasets import KarateClub, CitationFull, Planetoid
from dataclasses import dataclass
from torch_geometric.utils import to_undirected
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.datasets import GraphLandDataset
from torch_geometric.utils import subgraph
import copy
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42

In [ ]:
def build_negative_probs(edge_index: torch.Tensor,
    num_nodes: int, alpha: float = 0.75) -> torch.Tensor:
  """P_n(v) ∝ deg(v)^alpha, normalizada em [0,1]."""
  row, col = edge_index
  # Grafo não-direcionado: conta grau pela soma das duas pontas
  deg = torch.bincount(torch.cat([row, col]), minlength=num_nodes).float()
  weights = deg.clamp(min=1e-12).pow(alpha)   # evita prob=0 p/ nós isolados
  return weights / weights.sum()

def sample_negatives(probs: torch.Tensor,
    batch_size: int, Q: int) -> torch.Tensor:
  """Amostra Q negativos por par positivo → shape [B, Q]."""
  total = batch_size * Q
  # replacement=True necessário quando total > num_nodes
  idx = torch.multinomial(probs, num_samples=total, replacement=True)
  return idx.view(batch_size, Q)

def build_csr(edge_index : torch.Tensor, num_nodes:int, force_undirect = False):
  """Passar edge_denx em cpu"""
  if force_undirect:
    edge_index = to_undirected(edge_index, num_nodes=num_nodes)
  row, col = edge_index
  order = torch.argsort(row)
  col_sorted = col[order]
  if col_sorted.numel() == 0:
    return row_ptr, torch.zeros(0, dtype=torch.long, device=row.device)
  deg = torch.bincount(row, minlength=num_nodes)
  row_ptr = torch.zeros(num_nodes + 1, dtype=torch.long, device=row.device)
  row_ptr[1:] = torch.cumsum(deg, 0)
  
  return row_ptr, col_sorted

def sample_step(cur : torch.Tensor, row_ptr : torch.Tensor, col_idx : torch.Tensor):
  """cur: [B] → nxt: [B]. Um rand por nó, sem loop Python."""
  start = row_ptr[cur]
  end   = row_ptr[cur + 1]
  deg   = end - start   # grau de cada entrada de cada vertex idx em 'cur'
  deg_safe = deg.clamp(min=1)
  # print("deg",deg_safe)
  off = (torch.rand_like(deg, dtype=torch.float, device=start.device) * deg_safe).long() # offset sorteado
  # print("offset sample:", off)
  off = off.clamp(max=deg_safe - 1) # offset < deg_safe - 1
  idx = (start + off).clamp(max=col_idx.numel() - 1)
  nxt = col_idx[idx]
  return torch.where(deg == 0, cur, nxt) # se tiver aresta retorna nxt, se não, cur

def random_walks(u_batch : torch.Tensor, row_ptr : torch.Tensor, col_idx : torch.Tensor, walk_length : int, num_walks = 1):
  """[B, num_walks, walk_length+1]"""
  B, W = u_batch.numel(), num_walks
  walks = torch.empty(B, W, walk_length + 1,
    dtype=u_batch.dtype, device=u_batch.device)  # [B, W]
  walks[..., 0] = u_batch.unsqueeze(1)
  flat = u_batch.repeat_interleave(W)      # [B*W]
  for t in range(1, walk_length + 1):       # loop curto, tudo vetorizado dentro
      flat = sample_step(flat, row_ptr, col_idx)
      walks[:, :, t] = flat.view(B, W)
  return walks


def sample_positives(u_batch : torch.Tensor, row_ptr : torch.Tensor, 
    col_idx : torch.Tensor, walk_length=5, context_size=2, num_pos=1):
  """v coocorre com u em walk de comprimento fixo  [B, num_pos]."""
  assert context_size <= walk_length
  walks = random_walks(u_batch, row_ptr, col_idx, walk_length, num_pos)
  offsets = torch.randint(1, context_size + 1,
                          (u_batch.numel(), num_pos),
                          device=u_batch.device)
  return walks.gather(2, offsets.unsqueeze(-1),).squeeze(-1)


def rebuild_graph_data(data : Data, node_idx : torch.Tensor , edge_index : torch.Tensor):
  data_train = copy.copy(data)
  data_train.edge_index = edge_index
  data_train.x = data_train.x[node_idx]
  data_train.y = data_train.y[node_idx]
  return data

def aggr_sum(h : torch.Tensor, edge_index : torch.Tensor):
  out = torch.zeros(h.size(0), h.size(1), device=h.device, dtype=h.dtype)
  out.index_add_(0, edge_index[1], h[edge_index[0]])
  return out

def aggr_mean(h : torch.Tensor, edge_index : torch.Tensor):
  out = aggr_sum(h, edge_index)
  deg = torch.zeros(h.size(0),device=h.device)
  deg.index_add_(0,edge_index[1] ,torch.ones(edge_index.size(1),device=out.device))
  out = out / deg.clamp(min=1).unsqueeze(-1)
  return out

def get_aggr(aggr):
  if aggr == "sum":
        return aggr_sum
  elif aggr == "mean":
    return aggr_mean
  
  raise NotImplementedError()

class SageLayer(torch.nn.Module):
  def __init__(self, /,in_feat, out_feat, aggr, normalize, device):
    super().__init__()
    self.normalize = normalize
    self.aggr_func = get_aggr(aggr)
    self.aggr = aggr
    self.lin_root = torch.nn.Linear(in_features=in_feat,out_features=out_feat,bias=True, device=device)
    self.lin_neigh = torch.nn.Linear(in_features=in_feat,out_features=out_feat,bias=False, device=device)

  def forward(self, x, edge_index):
    neigh_aggr = self.aggr_func(x, edge_index)
    h = self.lin_root(x) + self.lin_neigh(neigh_aggr)
    h = F.relu(h)
    if self.normalize:
      h = F.normalize(h, p=2, dim=-1)
    return h

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class SageConfig:
    layers: int
    in_feat: int
    out_feat: int
    aggr: str 
    normalize: bool
    normalize_intern: bool
    device: Optional[torch.device]

    def build(self) -> torch.nn.ModuleList:
      """Constrói a lista de SageLayer conforme a configuração."""
      modules = []
      for i in range(self.layers):
        in_f = self.in_feat if i == 0 else self.out_feat
        out_f = self.out_feat
        # Só a última camada usa `normalize`; as internas usam `normalize_intern`
        norm = self.normalize if i == self.layers - 1 else self.normalize_intern
        modules.append(
          SageLayer(
            in_feat=in_f,
            out_feat=out_f,
            aggr=self.aggr,
            normalize=norm,
            device=self.device,
          )
        )
      return torch.nn.ModuleList(modules)

In [ ]:
epochs = 1000
batch = 1200
Q = 30
# sage_cfg = SageConfig(3,data.x.size(1),256,"mean",True,False, device)
# sagegnn = sage_cfg.build()


def train(model : torch.nn.Module, data: Data, opt : torch.optim.Optimizer, device):
  N = data.x.size(0)
  row_ptr, col_idx = build_csr(data.edge_index, N, force_undirect=True)
  data.x = data.x.to(device)
  model = model.to(device)

  model.train()
  for i in range(0, epochs):
    opt.zero_grad()
    z = model(data.x, data.edge_index)

    u_batch  = torch.randperm(N, device=data.device)
    neg_idx = sample_negatives(probs, N, Q)
    pos_idx = sample_positives(data.x,row_ptr,col_idx, 
      walk_length=2,context_size=2,num_pos=1)
  
    z_u, z_pos = z[u_batch], z[pos_idx]
    z_neg = z[neg_idx]  

    sig = F.sigmoid
    pos_score = (z_u*z_pos).sum()
    neg_score = torch.bmm(z_neg, z_u.unsqueeze(-1)).squeeze(-1) 
    loss = (-torch.log(torch.sigmoid(pos_score)).mean() - 
      - torch.log(torch.sigmoid(-neg_score)).mean())
    loss.backward()
    opt.step()



In [ ]:
input = torch.tensor([[1,2,3],
 [6,5,4],
 [1,2,3]])

print(input)
matrix = torch.tensor([1,2,3]).unsqueeze(-1)
print(matrix)

res = torch.bmm(input,matrix)
print(res)

tensor([[1, 2, 3],
        [6, 5, 4],
        [1, 2, 3]])
tensor([[1],
        [2],
        [3]])


RuntimeError: batch1 must be a 3D tensor

In [51]:
data = Planetoid("/home/milnner00/projects/gnn_recsys/data","cora").data
N = data.x.size(0)
N_train = int(N * 0.8)
N_val =  int(N * 0.1)
N_test = N - N_train - N_val



rgn = torch.Generator().manual_seed(seed)
idx = torch.randperm(N,generator=rgn)
splits = torch.split(idx, [N_train, N_val, N_test])
train_idx, val_idx, test_idx = splits


train_ei = subgraph(train_idx, data.edge_index, relabel_nodes=True)[0]
val_ei = subgraph(val_idx, data.edge_index, relabel_nodes=True)[0]
test_ei = subgraph(test_idx,data.edge_index, relabel_nodes=True)[0]


def rebuild_graph_data(data : Data, node_idx : torch.Tensor , edge_index : torch.Tensor):
  data_train = copy.copy(data)
  data_train.edge_index = edge_index
  data_train.x = data_train.x[node_idx]
  data_train.y = data_train.y[node_idx]
  return data_train

data_train = rebuild_graph_data(data, train_idx, train_ei)

/tmp/ipykernel_2817857/1738176985.py:1: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  data = Planetoid("/home/milnner00/projects/gnn_recsys/data","cora").data


In [52]:
data_train.edge_index

tensor([[1742,  738,  798,  ...,  210,  450,  832],
        [1415, 1415, 1415,  ..., 2163, 2163, 2163]])

In [53]:
# 1) Todo nó local referenciado existe em x
assert data_train.edge_index.max() < data_train.num_nodes, "edge_index OOB"

# 2) Cada aresta local mapeia para uma aresta real no grafo original
mapped = train_idx[data_train.edge_index]               # [2, E_sub] em IDs globais
orig   = set(map(tuple, data.edge_index.t().tolist()))
assert set(map(tuple, mapped.t().tolist())).issubset(orig), "aresta inválida"

# 3) Features batem com o subconjunto
assert torch.equal(data_train.x, data.x[train_idx])

In [56]:
print(data_train.edge_index)
print(mapped)
train_idx[1415]

tensor([[1742,  738,  798,  ...,  210,  450,  832],
        [1415, 1415, 1415,  ..., 2163, 2163, 2163]])
tensor([[ 633, 1862, 2582,  ...,  165, 1473, 2706],
        [   0,    0,    0,  ..., 2707, 2707, 2707]])


tensor(0)

In [39]:

# name pode ser "hm-categories", "web-topics", "artnet-exp", etc.
dataset = GraphLandDataset(root="/home/milnner00/projects/gnn_recsys/data", name="web-topics", split="THI")

train_data = dataset[0]  # grafo de treino
val_data   = dataset[1]  # grafo de validação (nós não vistos no treino)
test_data  = dataset[2]  # grafo de teste  (nós não vistos no treino nem na val)

KeyboardInterrupt: 

In [12]:
row, col = torch.tensor([0, 0, 1, 1, 2, 3,3]),\
          torch.tensor([1, 2, 2, 3, 3, 0,6])
r_max, c_max = row.max().item(), col.max().item()
N = (r_max if r_max > c_max else c_max) + 1
row + 2*N

batch_ei = torch.cat([
  row,   
  # row + N,
  # row + 2*N,

  col,
  # col + N,
  # col + 2*N
], dim = 0,).view(2, col.size(0))

print(batch_ei.shape)

print(batch_ei)
print()

indptr, indices = build_csr(batch_ei,batch_ei.max().item() + 1)
print(indptr)
print(indices)


torch.Size([2, 7])
tensor([[0, 0, 1, 1, 2, 3, 3],
        [1, 2, 2, 3, 3, 0, 6]])

tensor([ 0,  3,  6,  9, 13, 13, 13, 14])
tensor([1, 2, 3, 0, 2, 3, 0, 1, 3, 0, 1, 2, 6, 3])


In [20]:
u_batch = torch.tensor([0,0,0,3,5])
num_walks = 2
walk_length = 2
context_size = 2

"""→ [B, num_walks, walk_length+1]"""
B, W = u_batch.numel(), num_walks
walks = torch.empty(B, W, walk_length + 1,
  dtype=u_batch.dtype, device=u_batch.device)  # [B, W]
print(walks.shape)
print()
walks[..., 0] = u_batch.unsqueeze(1)
print(walks)
print()
flat = u_batch.repeat_interleave(W)       # [B*W]
print("row_ptr",indptr)
print("col_idx",indices)
print(flat)
for t in range(1, walk_length + 1):       # loop curto, tudo vetorizado dentro
    print("enter",flat)
    flat = sample_step(flat, indptr, indices)
    print("out:",flat)
    walks[:, :, t] = flat.view(B, W)
    

torch.Size([5, 2, 3])

tensor([[[                   0, -1142224136920881663,      403452685529856],
         [                   0, -1148417010752565197,  7061908201604473350]],

        [[                   0, -1146164948175359976, -6332456797172047360],
         [                   0,  7061908201604473344,   166100244928921869]],

        [[                   0,    67569391906980080,   166071267286385230],
         [                   0,    72706309992874242, -1148979745522651122]],

        [[                   3, -1136876215442851583,   144182190522047232],
         [                   3,   144119522771652672,   499969931008246801]],

        [[                   5,   284076424715698384,  4467808460222824960],
         [                   5, -1148416741027606527,  5188189583233322752]]])

row_ptr tensor([ 0,  3,  6,  9, 13, 13, 13, 14])
col_idx tensor([1, 2, 3, 0, 2, 3, 0, 1, 3, 0, 1, 2, 6, 3])
tensor([0, 0, 0, 0, 0, 0, 3, 3, 5, 5])
enter tensor([0, 0, 0, 0, 0, 0, 3, 3, 5, 5])
out:

In [24]:
offsets = torch.randint(
  1, 
  context_size + 1,
  (u_batch.numel(), num_walks),
  device=u_batch.device)

pos = walks.gather(2, offsets.unsqueeze(-1)).squeeze(-1)

print("oofsets:", offsets)
print("offsets_unsq:",offsets.unsqueeze(-1))
print("walks:", walks)
print(pos.shape)
print("pos:", pos.squeeze())

oofsets: tensor([[1, 1],
        [2, 1],
        [1, 1],
        [2, 2],
        [1, 1]])
offsets_unsq: tensor([[[1],
         [1]],

        [[2],
         [1]],

        [[1],
         [1]],

        [[2],
         [2]],

        [[1],
         [1]]])
walks: tensor([[[0, 2, 1],
         [0, 2, 0]],

        [[0, 3, 6],
         [0, 3, 0]],

        [[0, 3, 0],
         [0, 1, 3]],

        [[3, 1, 3],
         [3, 1, 0]],

        [[5, 5, 5],
         [5, 5, 5]]])
torch.Size([5, 2])
pos: tensor([[2, 2],
        [6, 3],
        [3, 1],
        [3, 0],
        [5, 5]])


In [ ]:
v = [1, 2, 2, 2, 3, 4, 5, 5, 5, 5]
v = torch.tensor(v).reshape((1,len(v)))

print("vetor:", v)
print(v.shape)
counts = torch.bincount(v[0], minlength=v.max())
print("counts:",counts)
probs = counts / v.size(1)
print("probs:", probs)
print("sum probs:", probs.sum())

samples = torch.multinomial(probs,10000, replacement=True)
count_p = torch.bincount(samples)

probs_p = count_p / 10000
print("samples:", probs_p )



tensor([[2, 2],
        [6, 3],
        [3, 1],
        [3, 0],
        [5, 5]])


In [ ]:
print(graph.get_summary)
graph.data


<bound method Dataset.get_summary of CoraFull()>


/tmp/ipykernel_2817857/2628678813.py:3: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  graph.data


Data(x=[19793, 8710], edge_index=[2, 126842], y=[19793])

In [22]:
data = graph.data

/tmp/ipykernel_2240843/4038582665.py:1: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  data = graph.data


In [24]:
data

Data(x=[19793, 8710], edge_index=[2, 126842], y=[19793])

In [5]:

# --- teste 1: single graph vs batch de 2 cópias ---
torch.manual_seed(0)
N, d = 4, 3
h_single = torch.randn(N, d)
ei_single = torch.tensor([[0, 1, 2, 3], [1, 2, 3, 0]])  # ciclo

out_single = aggr_mean(h_single, ei_single)

# Batch de 2 cópias, block-diagonal com offset
h_batch = torch.cat([h_single, h_single], dim=0)  # (8, 3)
ei_batch = torch.cat([ei_single, ei_single + N], dim=1)  # (2, 8)
out_batch = aggr_mean(h_batch, ei_batch)

assert torch.allclose(out_batch[:N], out_single), "Grafo 1 diverge"
assert torch.allclose(out_batch[N:], out_single), "Grafo 2 diverge"
print("✓ Batch block-diagonal preserva agregação por grafo")

# --- teste 2: bug antigo (torch.ones(N)) explodiria em E > N ---
# Aqui E = 4, N = 4 → E == N, então não pega. Force E > N:
ei_big = torch.tensor([[0, 1, 2, 3, 0, 1], [1, 2, 3, 0, 2, 3]])  # E=6 > N=4
out_ok = aggr_mean(h_single, ei_big)  # não crasha com bincount
print("✓ aggr_mean funciona com E > N")

✓ Batch block-diagonal preserva agregação por grafo
✓ aggr_mean funciona com E > N


tensor([[ 0.0248, -0.5383,  0.1465,  0.0194,  0.1435],
        [-0.2784, -0.0875, -0.0515,  0.0566, -0.2488],
        [-0.3132,  0.5139, -0.1732, -0.5692, -0.5419]],
       grad_fn=<PermuteBackward0>)


tensor([[-0.4433,  0.0549, -0.1914, -1.0796, -0.4633],
        [-0.4333, -0.0080, -0.2162, -0.4732, -0.3138],
        [-0.4433,  0.0549, -0.1914, -1.0796, -0.4633]],
       grad_fn=<AddmmBackward0>)

In [16]:
l = list(range(1,11))
print(l)
print()
print(l[:5]) # [1, 2, 3, 4, 5]
print()
print(l[2:6]) # [3, 4, 5, 6]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

[1, 2, 3, 4, 5]

[3, 4, 5, 6]
